# 02. 리뷰 재필터링: B 최종 asin_list 기준 (C, 2026-10-06)

**왜 다시 하나?** 9/30에 리뷰를 거를 때(`01_reviews_explore.ipynb`)는 B 결과가 없어서 원본 구매 CSV로 만든 **임시 ASIN 목록**을 썼다. 이제 B(구매 전처리)가 끝나서 최종 `data/processed/asin_list.csv`(13,817개)가 나왔으니, 리뷰를 이 목록 기준으로 맞춘다.

**리뷰 원본(13GB)은 다시 읽지 않는다.** 9/30에 만든 `reviews_filtered.parquet`(이미 6품목 ASIN·2018~2022로 걸러지고 중복 제거된 110만 건)에 B 목록을 다시 붙이기만 하면 된다. 두 목록의 ASIN이 같아서(13,817개) 빠지는 리뷰는 없고, 달라질 수 있는 건 두 품목에 걸친 ASIN `B005NIKG4E`의 품목뿐이다.

**실행 방법**: VS Code에서 이 노트북을 열고, 커널이 `.venv`인지 확인한 뒤 위쪽 **Run All**. 끝나면 **Ctrl+S로 저장**(출력이 저장돼야 결과가 GitHub에 남는다).

| 입력 | 출력 |
|---|---|
| `data/interim/filtered_reviews/reviews_filtered.parquet` (9/30 결과) | 같은 파일을 B 기준으로 다시 저장 (9/30 원본은 `reviews_filtered_0930.parquet`로 백업) |
| `data/processed/asin_list.csv` (B 최종) | `data/processed/weekly_reviews/weekly_reviews.csv` (6품목 × 261주) |
| | 품목별 리뷰 매칭률 표, 로그에 붙일 문장 |

## 1. 준비: 경로와 DuckDB 연결

- **DuckDB**: 큰 파일을 메모리에 다 올리지 않고 SQL로 바로 읽는 도구. 9/30에도 썼다.
- `SET TimeZone = 'UTC'`: 리뷰 시각(timestamp)은 UTC 기준이다. 안 맞추면 컴퓨터 시간대(한국, +9시간)로 날짜가 바뀌어 주가 하루씩 밀릴 수 있다.
- **백업**: 이 노트북은 `reviews_filtered.parquet`를 덮어쓴다. 그래서 처음 실행할 때 9/30 원본을 `reviews_filtered_0930.parquet`로 한 번 복사해 두고, 이후로는 **항상 백업본을 입력으로** 쓴다. 그래서 여러 번 다시 실행해도 결과가 같다.

In [ ]:
import shutil
from pathlib import Path
import duckdb

DATA = Path("../data")                                   # 노트북이 notebooks/ 안에 있어서 한 칸 위
REVIEWS = DATA / "interim/filtered_reviews/reviews_filtered.parquet"
BACKUP  = DATA / "interim/filtered_reviews/reviews_filtered_0930.parquet"
ASIN_LIST = DATA / "processed/asin_list.csv"
WEEKLY  = DATA / "processed/weekly_reviews/weekly_reviews.csv"

if not BACKUP.exists():                                   # 처음 한 번만 백업
    shutil.copy(REVIEWS, BACKUP)
    print("9/30 원본을 백업했습니다:", BACKUP)
else:
    print("백업본이 이미 있습니다. 백업본을 입력으로 씁니다:", BACKUP)

con = duckdb.connect()
con.sql("SET TimeZone = 'UTC'")
con.sql("SET preserve_insertion_order = false")         # 메모리 절약

## 2. B의 최종 ASIN 목록 불러오기

`asin_list.csv`는 B가 만든 표로, 컬럼은 `asin, product_type, purchase_rows`(그 ASIN의 구매 행 수)다.
- `all_varchar=true`: 모든 값을 문자로 읽는다. ASIN이 `0439738628`처럼 숫자로만 된 경우 숫자로 읽히면 앞의 0이 사라지기 때문이다.
- 확인: 행 수와 서로 다른 ASIN 수가 둘 다 13,817이어야 한다 (= ASIN 하나에 품목 하나).

In [ ]:
con.sql(f"""
    CREATE OR REPLACE TABLE asin_map AS
    SELECT asin, product_type, CAST(purchase_rows AS INTEGER) AS purchase_rows
    FROM read_csv('{ASIN_LIST.as_posix()}', all_varchar=true)
""")
n_rows, n_asin = con.sql("SELECT count(*), count(DISTINCT asin) FROM asin_map").fetchone()
print("asin_list 행 수:", n_rows, "/ 서로 다른 ASIN:", n_asin)
assert n_rows == n_asin, "ASIN 하나가 여러 품목에 들어가 있습니다. B 결과를 확인하세요."
con.sql("SELECT product_type, count(*) AS asins, sum(purchase_rows) AS purchase_rows FROM asin_map GROUP BY 1 ORDER BY 1").df()

## 3. 9/30 결과와 비교

9/30 리뷰(임시 목록 기준)에 B 목록을 **LEFT JOIN**(왼쪽 표의 행은 모두 남기고, 오른쪽에 짝이 있으면 붙임)해서 세 가지를 센다.
- `전체_리뷰`: 9/30 결과 리뷰 수 (1,107,639건이었다)
- `B목록에_있는_리뷰`: 그중 ASIN이 B 목록에 있는 리뷰. 두 목록이 같으면 전체와 같아야 한다.
- `품목이_바뀐_리뷰`: B 목록의 품목과 9/30에 붙인 품목이 다른 리뷰. B005NIKG4E 리뷰만 나올 수 있다.

In [ ]:
old = f"read_parquet('{BACKUP.as_posix()}')"
compare = con.sql(f"""
    SELECT
        count(*) AS 전체_리뷰,
        count(m.asin) AS B목록에_있는_리뷰,
        count(*) - count(m.asin) AS B목록에_없는_리뷰,
        sum(CASE WHEN m.asin IS NOT NULL AND m.product_type <> r.product_type THEN 1 ELSE 0 END) AS 품목이_바뀐_리뷰
    FROM {old} r
    LEFT JOIN asin_map m ON r.asin = m.asin
""").df()
compare

In [ ]:
# 품목이 바뀐 ASIN이 있다면 어떤 것인지 (9/30 품목 → B 품목)
con.sql(f"""
    SELECT r.asin, r.product_type AS 품목_0930, m.product_type AS 품목_B, count(*) AS 리뷰_수
    FROM {old} r JOIN asin_map m ON r.asin = m.asin
    WHERE r.product_type <> m.product_type
    GROUP BY 1, 2, 3
""").df()

## 4. B 목록 기준으로 리뷰 표 다시 만들기

- `JOIN`(짝이 있는 행만 남김): B 목록에 있는 ASIN의 리뷰만 남긴다.
- `r.* EXCLUDE (product_type)`: 9/30에 붙인 품목 열을 빼고, `m.product_type`(B 목록의 품목)으로 바꿔 끼운다.
- 결측 삭제·중복 제거·2018~2022 기간 필터는 9/30에 이미 했으므로 다시 하지 않는다 (입력이 그 결과물).
- 결과를 `reviews_filtered.parquet`에 저장한다 (백업은 그대로 남아 있음).

In [ ]:
con.sql(f"""
    CREATE OR REPLACE TABLE reviews AS
    SELECT r.* EXCLUDE (product_type), m.product_type
    FROM {old} r
    JOIN asin_map m ON r.asin = m.asin
""")
con.sql(f"COPY reviews TO '{REVIEWS.as_posix()}' (FORMAT parquet)")

summary = con.sql("""
    SELECT product_type,
           count(*) AS reviews,
           count(DISTINCT asin) AS asins,
           round(avg(rating), 2) AS avg_rating,
           sum(CASE WHEN text IS NULL OR trim(text) = '' THEN 1 ELSE 0 END) AS empty_text
    FROM reviews GROUP BY 1 ORDER BY 1
""").df()
print("재필터링 후 리뷰 수:", int(summary["reviews"].sum()))
summary

## 5. 주간 리뷰 집계 다시 만들기

`01` 노트북 셀 8과 같은 방법이다.
- **주 = 일요일 시작** (팀 기준, Google Trends와 같음). DuckDB의 `date_trunc('week', …)`는 월요일 시작이라, 날짜에 하루를 더해 월요일 기준으로 자른 뒤 하루를 빼면 그 주 일요일이 된다.
- **기준 표 = 6품목 × 261주** (2017-12-31 ~ 2022-12-25)를 먼저 만들고 집계를 LEFT JOIN한다. 리뷰가 없는 주도 빠지지 않게 하기 위해서다.
- 리뷰가 없는 주: `review_count`는 0, `avg_rating` 등 비율은 **비워 둔다**(0으로 채우면 "평점 0점"으로 오해됨).
- `share_1star`, `share_5star`: 1점·5점 비율 (리뷰 평점이 양 끝에 몰리는 J자 분포 확인용). `verified_share`: 인증 구매 리뷰 비율.

In [ ]:
weekly = con.sql("""
WITH weeks AS (
    SELECT CAST(w AS DATE) AS week
    FROM generate_series(DATE '2017-12-31', DATE '2022-12-25', INTERVAL 7 DAY) g(w)
),
types AS (SELECT DISTINCT product_type FROM asin_map),
agg AS (
    SELECT product_type,
           CAST(date_trunc('week', review_date + INTERVAL 1 DAY) - INTERVAL 1 DAY AS DATE) AS week,
           count(*) AS review_count,
           avg(rating) AS avg_rating,
           avg(CASE WHEN rating = 1 THEN 1.0 ELSE 0 END) AS share_1star,
           avg(CASE WHEN rating = 5 THEN 1.0 ELSE 0 END) AS share_5star,
           avg(CASE WHEN verified_purchase THEN 1.0 ELSE 0 END) AS verified_share
    FROM reviews GROUP BY 1, 2
)
SELECT w.week, t.product_type,
       coalesce(a.review_count, 0) AS review_count,
       round(a.avg_rating, 3) AS avg_rating,
       round(a.share_1star, 3) AS share_1star,
       round(a.share_5star, 3) AS share_5star,
       round(a.verified_share, 3) AS verified_share
FROM weeks w CROSS JOIN types t
LEFT JOIN agg a ON a.week = w.week AND a.product_type = t.product_type
ORDER BY t.product_type, w.week
""").df()

print("행 수:", len(weekly), "/ 주 수:", weekly["week"].nunique(), "/ 품목 수:", weekly["product_type"].nunique())
assert len(weekly) == 6 * 261, "6품목 × 261주가 아닙니다."
assert weekly["review_count"].sum() == summary["reviews"].sum(), "주간 합계가 전체 리뷰 수와 다릅니다 (기간 밖 리뷰가 섞였는지 확인)."

WEEKLY.parent.mkdir(parents=True, exist_ok=True)
weekly.to_csv(WEEKLY, index=False)
print("저장:", WEEKLY)
weekly.groupby("product_type")["review_count"].agg(["sum", "min", "max"])

## 6. 품목별 리뷰 매칭률 (발표 슬라이드용)

"구매 데이터에 나온 상품 중 몇 %가 리뷰 데이터에도 있나"를 두 기준으로 센다.
- **ASIN 기준** = 리뷰가 1건이라도 있는 ASIN ÷ 구매 데이터의 ASIN
- **구매 행 기준** = 리뷰가 있는 ASIN의 구매 행 합 ÷ 전체 구매 행 합. 많이 팔린 상품일수록 리뷰가 있을 가능성이 높아서 보통 ASIN 기준보다 높다.

주의: 여기서는 **2018~2022년에 쓰인 리뷰만** 센다 (리뷰 표가 그 기간으로 걸러져 있음). 명세 1장의 매칭률(64~96%)은 기간 제한 없이 리뷰 파일 전체로 센 값이라 조금 다를 수 있다.

In [ ]:
match = con.sql("""
    WITH has_review AS (SELECT DISTINCT asin FROM reviews)
    SELECT
        m.product_type,
        count(*) AS 구매_ASIN,
        count(h.asin) AS 리뷰있는_ASIN,
        round(100.0 * count(h.asin) / count(*), 1) AS ASIN_매칭률,
        sum(m.purchase_rows) AS 구매_행,
        sum(CASE WHEN h.asin IS NOT NULL THEN m.purchase_rows ELSE 0 END) AS 리뷰있는_구매_행,
        round(100.0 * sum(CASE WHEN h.asin IS NOT NULL THEN m.purchase_rows ELSE 0 END) / sum(m.purchase_rows), 1) AS 구매행_매칭률
    FROM asin_map m
    LEFT JOIN has_review h ON m.asin = h.asin
    GROUP BY 1 ORDER BY 1
""").df()
match

## 7. 로그에 붙일 문장 만들기

아래 셀이 출력하는 내용을 그대로 복사해서 `logs/preprocessing_log.md` 맨 아래에 붙인다.

In [ ]:
c = compare.iloc[0]
lines = [
    "### 2026-10-06 C. 리뷰 재필터링 (김지우)",
    "노트북: `notebooks/02_reviews_refilter.ipynb`",
    "",
    f"- B 최종 asin_list({n_asin:,}개)로 9/30 결과(reviews_filtered.parquet)를 다시 거름. 리뷰 원본은 다시 읽지 않음",
    f"- 9/30 리뷰 {int(c['전체_리뷰']):,}건 → B 목록에 있는 리뷰 {int(c['B목록에_있는_리뷰']):,}건 (빠진 리뷰 {int(c['B목록에_없는_리뷰']):,}건), 품목이 바뀐 리뷰 {int(c['품목이_바뀐_리뷰']):,}건",
    f"- 주간 집계 weekly_reviews.csv: {len(weekly):,}행 (6품목 × 261주)",
    "",
    "| product_type | 리뷰 수 | ASIN 수 | 평균 평점 | ASIN 매칭률 | 구매 행 매칭률 |",
    "|---|---|---|---|---|---|",
]
for _, s in summary.merge(match, on="product_type").iterrows():
    lines.append(f"| {s['product_type']} | {int(s['reviews']):,} | {int(s['asins']):,} | {s['avg_rating']:.2f} | "
                 f"{s['ASIN_매칭률']}% ({int(s['리뷰있는_ASIN']):,}/{int(s['구매_ASIN']):,}) | {s['구매행_매칭률']}% |")
lines.append("")
lines.append("매칭률은 2018~2022년 리뷰 기준.")
print("\n".join(lines))